# Ramalan penjualan Wijaya Kusuma untuk lead time dan proyeksi 30, 60, 90 hari

Notebook memprediksi **total unit terjual per cabang, kategori, dan kadar**. Ramalan dimulai besok dan berakhir pada hari ke-`horizon_hari`. Untuk ROP, horizon mengikuti lead time pengadaan. Untuk proyeksi penjualan, tersedia pilihan 30, 60, dan 90 hari. Data transaksi yang digunakan adalah data simulasi.

Gunakan runtime CPU Google Colab. Jalankan sel dari atas ke bawah dan pilih `Dataset_ML_Wijaya_Kusuma_Training.csv.gz` pada bagian **3. Muat dan periksa dataset** jika diminta.


## 1. Siapkan pustaka

Pasang LightGBM dan SciPy. Pandas dan NumPy membantu mengolah penjualan harian serta menghitung metrik.


In [1]:
%pip install -q lightgbm==4.6.0 scipy


In [2]:
from pathlib import Path
import json
import pickle
import platform
from importlib.metadata import version

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.signal import lfilter


## 2. Tentukan lead time, horizon proyeksi, dan jendela riwayat

- `LEAD_TIMES = (14,)` memakai lead time bawaan 14 hari. Contoh `(10, 14, 21)` melatih model untuk tiga lead time yang digunakan toko. Masukkan durasi dalam hari kalender.
- `PROJECTION_HORIZONS` menetapkan horizon untuk proyeksi penjualan 30, 60, dan 90 hari.
- `WINDOWS` menentukan jumlah hari ke belakang yang digunakan sebagai fitur penjualan. Jendela ini terpisah dari durasi ramalan.

Notebook melatih satu model LightGBM per durasi, termasuk jika durasi yang sama dipakai untuk lead time dan proyeksi. Semua cabang tutup pada hari H Idulfitri dan buka setiap hari selama 21 hari sebelum serta 21 hari sesudahnya.


In [3]:
# Tambahkan lead time yang benar-benar digunakan toko, misalnya (10, 14, 21).
LEAD_TIMES = (14,)
PROJECTION_HORIZONS = (30, 60, 90)
if not LEAD_TIMES or any(type(days) is not int or days <= 0 for days in LEAD_TIMES):
    raise ValueError("LEAD_TIMES harus berisi satu atau lebih jumlah hari kalender positif.")
HORIZONS = tuple(sorted(set(LEAD_TIMES) | set(PROJECTION_HORIZONS)))
WINDOWS = (7, 14, 30, 60, 90)
GROUP = ["kode_cabang", "kategori", "kadar"]
DATE = "tanggal_ramalan"
SALES = "penjualan_hari_ini"
PRICE = "harga_antam"
PRICE_CHANGE = "perubahan_harga_7_persen"
PERIODS = {
    "latih": ("2023-01-01", "2024-12-31"),
    "validasi": ("2025-01-01", "2025-09-30"),
    "uji": ("2025-10-01", "2026-07-31"),
}
EID_DATES = pd.to_datetime(["2023-04-22", "2024-04-10", "2025-03-31", "2026-03-21"])
CLOSED_WEEKDAY = {"PWG": 5, "MDR": 6, "SPH": 6, "DSA": None}
ARIMA_ORDERS = ((1, 0, 0), (1, 0, 1), (1, 1, 0), (0, 1, 1), (1, 1, 1))
CATEGORICAL = ["branch_code", "category_code"]


## 3. Muat dan periksa dataset

Pilih file penjualan harian yang sama dengan eksperimen 14 hari. Kode membaca data per cabang, kategori, dan kadar, lalu memeriksa kelengkapan tanggal, kesesuaian hari tutup, serta tanggal sumber harga ANTAM. Setelah selesai, sel menampilkan jumlah kelompok dan rentang tanggal yang terbaca.


In [4]:
from pathlib import Path
from tempfile import mkdtemp

import lightgbm
from google.colab import files

dataset_name = "Dataset_ML_Wijaya_Kusuma_Training.csv.gz"
paths = [Path("/content") / dataset_name, Path("/content/dataset_wijaya_kusuma") / dataset_name]
dataset_path = next((path for path in paths if path.is_file() and path.stat().st_size > 0), None)
if dataset_path is None:
    upload_dir = Path(mkdtemp(prefix="wijaya_kusuma_", dir="/content"))
    uploaded = files.upload(target_dir=str(upload_dir))
    candidates = sorted(upload_dir.glob("*.csv.gz")) + sorted(upload_dir.glob("*.csv"))
    if len(candidates) != 1 or candidates[0].stat().st_size == 0:
        raise ValueError("Unggah tepat satu dataset CSV atau CSV.GZ yang dipakai pada pelatihan 14 hari.")
    dataset_path = candidates[0]
    del uploaded

print(f"Dataset: {dataset_path.name} | LightGBM: {lightgbm.__version__}")


Dataset: Dataset_ML_Wijaya_Kusuma_Training.csv.gz | LightGBM: 4.6.0


In [5]:
def load_panel(path):
    columns = [*GROUP, DATE, SALES, PRICE, PRICE_CHANGE, "tanggal_sumber_harga"]
    frame = pd.read_csv(path, usecols=columns)
    frame[DATE] = pd.to_datetime(frame[DATE], format="%Y-%m-%d")
    frame["tanggal_sumber_harga"] = pd.to_datetime(
        frame["tanggal_sumber_harga"], format="%Y-%m-%d"
    )
    if frame[[*GROUP, DATE, SALES]].isna().any().any():
        raise ValueError("Identitas kelompok, tanggal, dan penjualan harian wajib terisi.")
    for name in GROUP[:2]:
        frame[name] = frame[name].astype(str).str.strip().str.upper()
    karat = pd.to_numeric(frame["kadar"].astype(str).str.upper().str.removesuffix("K"))
    if not (karat.between(8, 24) & karat.mod(1).eq(0)).all():
        raise ValueError("Kadar harus berupa bilangan bulat dalam rentang 8–24.")
    frame["kadar"] = karat.astype(int)
    if not np.isfinite(frame[SALES]).all() or not (
        frame[SALES].ge(0) & frame[SALES].mod(1).eq(0)
    ).all():
        raise ValueError("Penjualan harian harus berupa jumlah unit nonnegatif.")
    if frame["tanggal_sumber_harga"].gt(frame[DATE]).any():
        raise ValueError("Tanggal sumber harga melewati tanggal ramalan.")
    if not set(frame["kode_cabang"].unique()).issubset(CLOSED_WEEKDAY):
        raise ValueError("Jadwal cabang belum tersedia.")
    series = frame.pivot(index=GROUP, columns=DATE, values=SALES).sort_index(axis=1)
    dates = series.columns
    if not dates.equals(pd.date_range(dates.min(), dates.max(), name=DATE)):
        raise ValueError("Ada tanggal yang hilang.")
    if series.isna().any().any():
        raise ValueError("Setiap kelompok perlu memiliki penjualan harian lengkap.")
    if dates[-1] < pd.Timestamp(PERIODS["uji"][1]):
        raise ValueError("Riwayat belum mencakup seluruh periode uji.")
    values = series.to_numpy(dtype=float)
    identities = series.index.to_frame(index=False)
    offsets = (dates.to_numpy()[:, None] - EID_DATES.to_numpy()[None, :]) / np.timedelta64(1, "D")
    holiday = (offsets == 0).any(axis=1)
    extended = ((np.abs(offsets) <= 21) & (offsets != 0)).any(axis=1)
    pre_eid = ((offsets >= -21) & (offsets <= -1)).any(axis=1)
    schedules = {
        branch: ~holiday & (extended | (dates.dayofweek.to_numpy() != weekday))
        for branch, weekday in CLOSED_WEEKDAY.items()
    }
    opened = np.stack([schedules[code] for code in identities["kode_cabang"]])
    if values[~opened].any():
        raise ValueError("Ditemukan penjualan pada tanggal cabang tutup.")
    daily_price = frame[[DATE, PRICE, PRICE_CHANGE]].drop_duplicates()
    if daily_price[DATE].duplicated().any():
        raise ValueError("Harga acuan berbeda untuk tanggal yang sama.")
    daily_price = daily_price.set_index(DATE).reindex(dates)
    if not np.isfinite(daily_price[[PRICE, PRICE_CHANGE]].stack()).all():
        raise ValueError("Fitur harga harus berupa angka terbatas atau kosong.")
    if daily_price[PRICE].dropna().le(0).any():
        raise ValueError("Harga ANTAM harus positif.")
    return values, dates, identities, opened, pre_eid, daily_price


values, dates, identities, opened, pre_eid, daily_price = load_panel(dataset_path)
print(f"Kelompok: {len(identities):,} | Hari: {len(dates):,}")
print(f"Rentang dataset: {dates[0].date()} sampai {dates[-1].date()}")


Kelompok: 340 | Hari: 1,412
Rentang dataset: 2023-01-01 sampai 2026-11-12


## 4. Bagi dataset menjadi latih, validasi, dan uji

Pembagian dilakukan berdasarkan urutan tanggal. Target setiap observasi harus selesai di dalam periode yang ditetapkan.

| Periode data | Fungsi |
| --- | --- |
| Januari 2023–Desember 2024 | Latih parameter model. |
| Januari–September 2025 | Pilih kombinasi fitur LightGBM, jendela SMA/WMA, dan orde ARIMA. |
| Oktober 2025–Juli 2026 | Uji akurasi metode pada data yang disisihkan untuk evaluasi akhir. |

Untuk proyeksi 30, 60, dan 90 hari, tanggal ramalan pada validasi dan uji sama dengan eksperimen sebelumnya. Untuk lead time tambahan, tanggal ramalan terakhir ditentukan oleh durasinya agar seluruh penjualan aktual berada di periode yang benar. Semua metode dibandingkan pada tanggal dan kelompok yang sama dalam satu horizon. Gunakan MAE untuk membandingkan metode pada horizon yang sama.


In [6]:
def origin_indexes(dates, horizon, split):
    start, end = map(pd.Timestamp, PERIODS[split])
    # Pertahankan tanggal validasi dan uji dari eksperimen proyeksi sebelumnya.
    cutoff_horizon = horizon
    if split != "latih" and horizon in PROJECTION_HORIZONS:
        cutoff_horizon = max(PROJECTION_HORIZONS)
    eligible = (dates >= start) & (dates + pd.Timedelta(days=cutoff_horizon) <= end)
    return np.flatnonzero(eligible & (np.arange(len(dates)) >= max(WINDOWS) - 1))


from IPython.display import display

split_origins = {
    horizon: {split: origin_indexes(dates, horizon, split) for split in PERIODS}
    for horizon in HORIZONS
}
for horizon, splits in split_origins.items():
    for split, indices in splits.items():
        if len(indices) == 0:
            raise ValueError(f"Horizon {horizon} hari tidak memiliki tanggal ramalan pada data {split}.")
split_summary = pd.DataFrame([
    {
        "horizon_hari": horizon,
        "bagian": split,
        "tanggal_ramalan_awal": dates[indices[0]].date(),
        "tanggal_ramalan_akhir": dates[indices[-1]].date(),
        "tanggal_target_terakhir": (dates[indices[-1]] + pd.Timedelta(days=horizon)).date(),
        "jumlah_tanggal_ramalan": len(indices),
        "jumlah_observasi": len(indices) * len(identities),
    }
    for horizon, splits in split_origins.items()
    for split, indices in splits.items()
])
display(split_summary)


,horizon_hari,bagian,tanggal_ramalan_awal,tanggal_ramalan_akhir,tanggal_target_terakhir,jumlah_tanggal_ramalan,jumlah_observasi
0,14,latih,2023-03-31,2024-12-17,2024-12-31,628,213520
1,14,validasi,2025-01-01,2025-09-16,2025-09-30,259,88060
2,14,uji,2025-10-01,2026-07-17,2026-07-31,290,98600
3,30,latih,2023-03-31,2024-12-01,2024-12-31,612,208080
4,30,validasi,2025-01-01,2025-07-02,2025-08-01,183,62220
5,30,uji,2025-10-01,2026-05-02,2026-06-01,214,72760
6,60,latih,2023-03-31,2024-11-01,2024-12-31,582,197880
7,60,validasi,2025-01-01,2025-07-02,2025-08-31,183,62220
8,60,uji,2025-10-01,2026-05-02,2026-07-01,214,72760
9,90,latih,2023-03-31,2024-10-02,2024-12-31,552,187680


## 5. Buat fitur dan target penjualan

Untuk tanggal ramalan `t`, fitur riwayat menggunakan transaksi sampai `t`. Jadwal cabang dan hari menjelang Idulfitri dihitung dari kalender, sedangkan fitur ANTAM memakai harga yang sudah tersedia pada `t`. Target pelatihan menjumlahkan unit yang benar-benar terjual dari `t+1` hingga `t+horizon_hari`.


In [7]:
def prefix_sum(values):
    return np.pad(np.cumsum(values, axis=-1), [(0, 0)] * (values.ndim - 1) + [(1, 0)])


def feature_matrix(values, identities, opened, pre_eid, daily_price, horizon, origins, levels):
    n_groups = len(identities)
    n_dates = len(origins)
    sales = prefix_sum(values)
    opened_prefix = prefix_sum(opened)
    eid_prefix = prefix_sum(pre_eid)
    time_indexes = np.arange(values.shape[1])
    last_sale = np.maximum.accumulate(np.where(values > 0, time_indexes, -1), axis=1)
    last_sale = last_sale[:, origins]
    elapsed = np.where(last_sale < 0, np.nan, origins - last_sale)
    data = {
        "branch_code": np.repeat(identities["kode_cabang"].map(levels["branch_code"]), n_dates),
        "category_code": np.repeat(identities["kategori"].map(levels["category_code"]), n_dates),
        "karat": np.repeat(identities["kadar"].to_numpy(), n_dates),
    }
    for n in WINDOWS:
        data[f"sales_{n}d"] = (sales[:, origins + 1] - sales[:, origins + 1 - n]).ravel()
    data["days_since_last_sale"] = elapsed.ravel()
    data[f"open_days_next_{horizon}"] = (
        opened_prefix[:, origins + horizon + 1] - opened_prefix[:, origins + 1]
    ).ravel()
    data[f"pre_eid_days_next_{horizon}"] = np.tile(
        eid_prefix[origins + horizon + 1] - eid_prefix[origins + 1], n_groups
    )
    data["antam_idr_per_gram"] = np.tile(daily_price[PRICE].to_numpy()[origins], n_groups)
    data["antam_change_7d_pct"] = np.tile(daily_price[PRICE_CHANGE].to_numpy()[origins], n_groups)
    actual = (sales[:, origins + horizon + 1] - sales[:, origins + 1]).ravel()
    return pd.DataFrame(data), actual


levels = {
    "branch_code": {v: i for i, v in enumerate(sorted(identities["kode_cabang"].unique()))},
    "category_code": {v: i for i, v in enumerate(sorted(identities["kategori"].unique()))},
}
parts_by_horizon = {
    horizon: {
        split: feature_matrix(values, identities, opened, pre_eid, daily_price, horizon, indices, levels)
        for split, indices in splits.items()
    }
    for horizon, splits in split_origins.items()
}


## 6. Tentukan metrik evaluasi

MAE mengukur rata-rata selisih absolut prediksi terhadap penjualan aktual dalam unit. RMSE memberi pengaruh lebih besar pada galat besar. Bias negatif berarti ramalan cenderung lebih rendah daripada penjualan aktual. Pemilihan model memakai MAE validasi.


In [8]:
def score(actual, prediction):
    delta = np.asarray(prediction) - np.asarray(actual)
    return {
        "rows": int(delta.size), "mae": float(np.abs(delta).mean()),
        "rmse": float(np.sqrt(np.square(delta).mean())), "bias": float(delta.mean()),
    }


## 7. Hitung prediksi SMA dan WMA

SMA memberi bobot sama pada setiap hari dalam jendela riwayat, sedangkan WMA memberi bobot lebih besar pada hari terbaru. Masing-masing memakai jendela 30, 60, dan 90 hari. Rata-rata penjualan harian dikalikan dengan horizon untuk memperoleh prediksi total unit terjual.


In [9]:
def sma_wma_forecasts(values):
    totals = prefix_sum(values)
    widths = np.arange(1, values.shape[1] + 1)
    weighted = prefix_sum(values * widths)
    forecasts = {}
    for n in (30, 60, 90):
        endpoints = np.arange(n, values.shape[1] + 1)
        sum_y = totals[:, endpoints] - totals[:, endpoints - n]
        sum_wy = weighted[:, endpoints] - weighted[:, endpoints - n]
        forecasts[f"SMA{n}"] = sum_y / n
        forecasts[f"WMA{n}"] = (sum_wy - (endpoints - n) * sum_y) / (n * (n + 1) / 2)
    return forecasts


def moving_average_predictions(classical, origins, horizon):
    return {
        name: daily[:, origins - int(name[3:]) + 1].ravel() * horizon
        for name, daily in classical.items()
    }


classical = sma_wma_forecasts(values)
selected_moving_average = {}
for horizon in HORIZONS:
    valid_y = parts_by_horizon[horizon]["validasi"][1]
    valid_ma = moving_average_predictions(classical, split_origins[horizon]["validasi"], horizon)
    best_sma, best_wma = (
        min((name for name in valid_ma if name.startswith(family)),
            key=lambda name: score(valid_y, valid_ma[name])["mae"])
        for family in ("SMA", "WMA")
    )
    selected_moving_average[horizon] = (best_sma, best_wma)
    print(f"{horizon} hari | Pilihan validasi: {best_sma} dan {best_wma}")


14 hari | Pilihan validasi: SMA60 dan WMA90
30 hari | Pilihan validasi: SMA90 dan WMA90
60 hari | Pilihan validasi: SMA90 dan WMA90
90 hari | Pilihan validasi: SMA90 dan WMA90


## 8. Latih dan pilih orde ARIMA

Estimasi kandidat ARIMA untuk setiap kombinasi cabang, kategori, dan kadar. Koefisien dihitung dari data latih. Saat tanggal ramalan bergeser, ARIMA memperbarui keadaan model dengan penjualan yang sudah terjadi sampai tanggal itu. Prediksi harian dijumlahkan menjadi total per horizon. Orde terbaik ditentukan dari MAE validasi untuk setiap horizon.


In [10]:
def fit_arima(train, order):
    """Fit a stationary/invertible ARIMA(p,d,q) with conditional least squares."""
    p, d, q = order
    series = np.diff(train) if d else train
    center = 0.0 if d else float(series.mean())
    centered = series - center
    if np.all(centered == 0):
        return center, 0.0, 0.0

    def coeffs(params):
        return (float(params[0]) if p else 0.0, float(params[p]) if q else 0.0)

    def objective(params):
        phi, theta = coeffs(params)
        errors = lfilter([1.0, -phi], [1.0, theta], centered)
        return float(np.mean(errors[1:] ** 2))

    bounds = [(-0.98, 0.98)] * (p + q)
    result = minimize(objective, np.zeros(p + q), method="L-BFGS-B", bounds=bounds)
    if not result.success:
        result = minimize(objective, np.zeros(p + q), method="Powell", bounds=bounds)
    if not result.success or not np.isfinite(result.fun):
        raise RuntimeError(f"ARIMA{order} tidak konvergen.")
    return center, *coeffs(result.x)


def rolling_arima(values, dates, origins, order):
    """Keep fitted parameters fixed. Update each forecast state with actuals through t."""
    p, d, q = order
    train_end = dates.searchsorted(pd.Timestamp(PERIODS["latih"][1]), side="right")
    if train_end == 0 or int(origins.min()) < train_end:
        raise ValueError("Tanggal ramalan ARIMA harus sesudah periode estimasi.")
    predictions = np.empty((len(values), len(origins), max(HORIZONS)))
    steps = np.arange(max(HORIZONS))
    for index, sales in enumerate(values):
        center, phi, theta = fit_arima(sales[:train_end], order)
        series = np.diff(sales) if d else sales
        centered = series - center
        errors = lfilter([1.0, -phi], [1.0, theta], centered)
        state = origins - d
        first = phi * centered[state] + theta * errors[state]
        ahead = center + first[:, None] * np.power(phi, steps)
        if d:
            ahead = sales[origins, None] + np.cumsum(ahead, axis=1)
        predictions[index] = np.cumsum(np.maximum(ahead, 0), axis=1)
    return predictions


evaluation_splits = ("validasi", "uji")
combined = np.unique(np.concatenate([
    split_origins[horizon][split] for horizon in HORIZONS for split in evaluation_splits
]))
arima = {}
for order in ARIMA_ORDERS:
    print(f"Menghitung ARIMA{order} untuk {len(identities)} kelompok", flush=True)
    cumulative = rolling_arima(values, dates, combined, order)
    arima[order] = {
        horizon: {
            split: cumulative[:, np.searchsorted(combined, split_origins[horizon][split]), horizon - 1].ravel()
            for split in evaluation_splits
        }
        for horizon in HORIZONS
    }
    del cumulative


Menghitung ARIMA(1, 0, 0) untuk 340 kelompok
Menghitung ARIMA(1, 0, 1) untuk 340 kelompok
Menghitung ARIMA(1, 1, 0) untuk 340 kelompok
Menghitung ARIMA(0, 1, 1) untuk 340 kelompok
Menghitung ARIMA(1, 1, 1) untuk 340 kelompok


In [11]:
chosen_arima = {}
order_rows = []
for horizon in HORIZONS:
    parts = parts_by_horizon[horizon]
    valid_y = parts["validasi"][1]
    arima_candidates = {
        order: score(valid_y, arima[order][horizon]["validasi"])
        for order in ARIMA_ORDERS
    }
    chosen_order = min(ARIMA_ORDERS, key=lambda order: arima_candidates[order]["mae"])
    best_arima = f"ARIMA{chosen_order}"
    for order, metrics in arima_candidates.items():
        order_rows.append({
            "horizon_hari": horizon, "split": "validasi", "metode": f"ARIMA{order}",
            "terpilih_pada_validasi": order == chosen_order, **metrics,
        })
    chosen_arima[horizon] = (chosen_order, best_arima)
    print(f"{horizon} hari | Pilihan validasi: {best_arima}")


14 hari | Pilihan validasi: ARIMA(1, 0, 1)
30 hari | Pilihan validasi: ARIMA(1, 0, 1)
60 hari | Pilihan validasi: ARIMA(1, 0, 1)
90 hari | Pilihan validasi: ARIMA(1, 0, 1)


## 9. Latih dan pilih model LightGBM

Latih empat varian fitur pada setiap horizon. Varian menggunakan riwayat saja, riwayat dan jadwal, tambahan Idulfitri, serta tambahan ANTAM. Pilih varian dengan MAE validasi terendah untuk disimpan sebagai model horizon tersebut.


In [12]:
def train_lightgbm(parts, horizon):
    import lightgbm as lgb

    base = [*CATEGORICAL, "karat", *(f"sales_{n}d" for n in WINDOWS), "days_since_last_sale"]
    schedule = base + [f"open_days_next_{horizon}"]
    calendar = schedule + [f"pre_eid_days_next_{horizon}"]
    variants = {
        "LightGBM_riwayat": base,
        "LightGBM_riwayat_jadwal": schedule,
        "LightGBM_riwayat_jadwal_idulfitri": calendar,
        "LightGBM_riwayat_jadwal_idulfitri_ANTAM": calendar
        + ["antam_idr_per_gram", "antam_change_7d_pct"],
    }
    models = {}
    val_x, val_y = parts["validasi"]
    for name, columns in variants.items():
        train_x, train_y = parts["latih"]
        train_set = lgb.Dataset(train_x[columns], label=train_y, categorical_feature=CATEGORICAL)
        valid_set = lgb.Dataset(val_x[columns], label=val_y, reference=train_set)
        model = lgb.train(
            {
                "objective": "poisson", "metric": "mae", "learning_rate": 0.045,
                "num_leaves": 47, "min_data_in_leaf": 140, "lambda_l2": 1.0,
                "seed": 2026, "num_threads": 4, "verbosity": -1,
                "deterministic": True, "force_col_wise": True,
            },
            train_set, num_boost_round=800, valid_sets=[valid_set],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        models[name] = (model, columns, score(val_y, model.predict(val_x[columns], num_threads=4)))
        print(f"Horizon {horizon} | {name} | MAE validasi {models[name][2]['mae']:.4f}", flush=True)
    return models


models_by_horizon = {}
selected_lightgbm = {}
for horizon in HORIZONS:
    models = train_lightgbm(parts_by_horizon[horizon], horizon)
    models_by_horizon[horizon] = models
    selected_lightgbm[horizon] = min(models, key=lambda name: models[name][2]["mae"])
    print(f"{horizon} hari | Pilihan validasi: {selected_lightgbm[horizon]}")


Horizon 14 | LightGBM_riwayat | MAE validasi 1.0243
Horizon 14 | LightGBM_riwayat_jadwal | MAE validasi 0.9015
Horizon 14 | LightGBM_riwayat_jadwal_idulfitri | MAE validasi 0.8585
Horizon 14 | LightGBM_riwayat_jadwal_idulfitri_ANTAM | MAE validasi 0.8799
14 hari | Pilihan validasi: LightGBM_riwayat_jadwal_idulfitri
Horizon 30 | LightGBM_riwayat | MAE validasi 1.6405
Horizon 30 | LightGBM_riwayat_jadwal | MAE validasi 1.1915
Horizon 30 | LightGBM_riwayat_jadwal_idulfitri | MAE validasi 1.1180
Horizon 30 | LightGBM_riwayat_jadwal_idulfitri_ANTAM | MAE validasi 1.1580
30 hari | Pilihan validasi: LightGBM_riwayat_jadwal_idulfitri
Horizon 60 | LightGBM_riwayat | MAE validasi 2.1888
Horizon 60 | LightGBM_riwayat_jadwal | MAE validasi 1.4329
Horizon 60 | LightGBM_riwayat_jadwal_idulfitri | MAE validasi 1.3387
Horizon 60 | LightGBM_riwayat_jadwal_idulfitri_ANTAM | MAE validasi 1.3905
60 hari | Pilihan validasi: LightGBM_riwayat_jadwal_idulfitri
Horizon 90 | LightGBM_riwayat | MAE validasi 2.68

## 10. Bandingkan hasil pada data uji

Bandingkan SMA, WMA, ARIMA, semua varian LightGBM, dan gabungan WMA dengan LightGBM 50:50. Semua metode memakai kelompok dan tanggal ramalan yang sama dalam setiap horizon, termasuk lead time 14 hari. Untuk menilai prediksi penjualan yang dipakai dalam ROP 14 hari, lihat hasil pada `horizon_hari = 14`. Tabel menampilkan peringkat MAE uji untuk masing-masing horizon.


In [13]:
rows = []
prediction_examples = []
for horizon in HORIZONS:
    parts = parts_by_horizon[horizon]
    best_sma, best_wma = selected_moving_average[horizon]
    chosen_order, best_arima = chosen_arima[horizon]
    models = models_by_horizon[horizon]
    chosen_lgb = selected_lightgbm[horizon]
    for split in evaluation_splits:
        inds = split_origins[horizon][split]
        matrix, actual = parts[split]
        pred = moving_average_predictions(classical, inds, horizon)
        pred[best_arima] = arima[chosen_order][horizon][split]
        for name, (model, columns, _) in models.items():
            pred[name] = np.maximum(0, model.predict(matrix[columns], num_threads=4))
        if chosen_lgb is not None:
            pred[f"Gabungan_{best_wma}_LightGBM_50_50"] = (pred[best_wma] + pred[chosen_lgb]) / 2
        if split == "uji":
            example_dates = dates[inds[[0, -1]]]
            examples = identities.loc[identities.index.repeat(len(example_dates))].reset_index(drop=True)
            examples["horizon_hari"] = horizon
            examples["tanggal_ramalan"] = np.tile(example_dates, len(identities))
            examples["target_mulai"] = examples["tanggal_ramalan"] + pd.Timedelta(days=1)
            examples["target_akhir"] = examples["tanggal_ramalan"] + pd.Timedelta(days=horizon)
            examples["unit_aktual"] = actual.reshape(len(identities), len(inds))[:, [0, -1]].ravel()
            for family, method in (("sma", best_sma), ("wma", best_wma), ("arima", best_arima),
                                   ("lightgbm", chosen_lgb)):
                if method is not None:
                    examples[f"metode_{family}"] = method
                    examples[f"prediksi_{family}"] = (
                        pred[method].reshape(len(identities), len(inds))[:, [0, -1]].ravel()
                    )
            prediction_examples.append(examples)
        for method, predictions in pred.items():
            rows.append({
                "horizon_hari": horizon, "split": split, "metode": method,
                "terpilih_pada_validasi": method in (best_sma, best_wma, best_arima, chosen_lgb),
                "tanggal_ramalan_awal": dates[inds[0]].date().isoformat(),
                "tanggal_ramalan_akhir": dates[inds[-1]].date().isoformat(),
                "akhir_target_terakhir": (dates[inds[-1]] + pd.Timedelta(days=horizon)).date().isoformat(),
                **score(actual, predictions),
            })

results = pd.DataFrame(rows).sort_values(["horizon_hari", "split", "mae"])


In [14]:
test = results.loc[results["split"].eq("uji")].copy()
test["peringkat_mae"] = test.groupby("horizon_hari")["mae"].rank(method="min").astype(int)
assert test.groupby("horizon_hari")["rows"].nunique().eq(1).all()
display(test[["horizon_hari", "peringkat_mae", "metode", "rows", "mae", "rmse", "bias"]]
        .round({"mae": 4, "rmse": 4, "bias": 4}).reset_index(drop=True))


,horizon_hari,peringkat_mae,metode,rows,mae,rmse,bias
0,14,1,LightGBM_riwayat_jadwal_idulfitri,98600,0.8599,1.1262,-0.0320
1,14,2,LightGBM_riwayat_jadwal_idulfitri_ANTAM,98600,0.8714,1.1457,-0.0630
2,14,3,LightGBM_riwayat_jadwal,98600,0.8853,1.1678,-0.0334
3,14,4,LightGBM_riwayat,98600,1.0059,1.3942,-0.0877
4,14,5,Gabungan_WMA90_LightGBM_50_50,98600,1.1508,1.4702,-0.0103
5,14,6,"ARIMA(1, 0, 1)",98600,1.5025,1.9017,-0.0794
6,14,7,SMA60,98600,1.5955,2.0237,0.0129
7,14,8,SMA90,98600,1.6570,2.0923,0.0091
8,14,9,WMA90,98600,1.7076,2.1463,0.0113
9,14,10,WMA60,98600,1.8002,2.2528,0.0116


## 11. Simpan model, metadata, dan hasil evaluasi

Dengan konfigurasi bawaan, ZIP memuat `lightgbm_14d.pkl`, `lightgbm_30d.pkl`, `lightgbm_60d.pkl`, dan `lightgbm_90d.pkl`. File model tambahan dibuat sesuai `LEAD_TIMES`. Metadata mencatat fitur, versi paket, dan fungsi setiap model.

Notebook membuka kembali model dan memeriksa konsistensi prediksi. Ekspor juga berisi tabel MAE, RMSE, dan bias, seleksi orde ARIMA, contoh prediksi, serta modul Python dan contoh data untuk pemanggilan dari Flask.


In [15]:
def save_lightgbm_model(horizon, models, chosen_lgb, parts, output):
    valid_y = parts["validasi"][1]
    model, columns, _ = models[chosen_lgb]
    metadata = {
        "schema_version": "multihorizon-1",
        "model_version": f"wk-lightgbm-{horizon}d-v2",
        "forecast_uses": [
            name for name, durations in (("lead_time", LEAD_TIMES), ("projection", PROJECTION_HORIZONS))
            if horizon in durations
        ],
        "algorithm": "LightGBM", "horizon_days": horizon,
        "timezone": "Asia/Jakarta", "target": f"Total unit terjual t+1 sampai t+{horizon}.",
        "history_days": max(WINDOWS), "feature_set": chosen_lgb,
        "feature_names": columns, "category_levels": levels,
        "known_karat": sorted(int(k) for k in identities["kadar"].unique()),
        "nullable_features": [
            k for k in ("days_since_last_sale", "antam_idr_per_gram", "antam_change_7d_pct")
            if k in columns
        ],
        "python_version": platform.python_version(),
        "package_versions": {k: version(k) for k in ("lightgbm", "numpy", "pandas", "scipy")},
        "training_transactions": "sintetis", "periods": PERIODS,
        "days_since_last_sale_missing": "Gunakan NaN jika belum ada penjualan yang tercatat.",
        "training_rows": len(parts["latih"][1]), "validation_rows": len(valid_y),
        "best_iteration": model.best_iteration,
    }
    model_path = output / f"lightgbm_{horizon}d.pkl"
    with model_path.open("wb") as handle:
        pickle.dump({"model": model, "metadata": metadata}, handle, protocol=4)
    with model_path.open("rb") as handle:
        loaded = pickle.load(handle)
    test_input = parts["uji"][0][columns].iloc[:3]
    np.testing.assert_allclose(
        loaded["model"].predict(test_input, num_threads=1),
        model.predict(test_input, num_threads=1),
    )
    (output / f"metadata_{horizon}d.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )


output_dir = Path(mkdtemp(prefix="hasil_wijaya_kusuma_", dir="/content"))
for horizon in HORIZONS:
    save_lightgbm_model(
        horizon, models_by_horizon[horizon], selected_lightgbm[horizon], parts_by_horizon[horizon], output_dir
    )

results.to_csv(output_dir / "Evaluasi_MAE_Lead_Time_dan_Proyeksi.csv", index=False)
pd.DataFrame(order_rows).sort_values(["horizon_hari", "mae"]).to_csv(
    output_dir / "Seleksi_Orde_ARIMA_Lead_Time_dan_Proyeksi.csv", index=False
)
pd.concat(prediction_examples, ignore_index=True).to_csv(output_dir / "Contoh_Prediksi_Uji.csv", index=False)


## 12. Contoh ramalan selama lead time untuk Flask dan ROP

Gunakan `load_lead_time_model(folder_model, lead_time_days)` dan `predict_lead_time(bundle, forecast_date, features)` dari modul yang diekspor. Flask dapat menyiapkan fitur berdasarkan `feature_names` dan `category_levels` di metadata, dengan data penjualan sampai tanggal ramalan. Fitur jadwal dan Idulfitri dihitung untuk periode yang diminta. Tanggal ramalan memakai format `YYYY-MM-DD`.

Output memuat tanggal mulai, tanggal akhir, lead time, dan `predicted_units` sebagai **total unit terjual selama lead time**. Laravel dapat menghitung `ceil(predicted_units + safety_stock)` untuk ROP dengan stok pengaman yang sudah dikonfigurasi. Jika model untuk lead time yang diminta belum dilatih, Flask dapat mengembalikan pesan dari modul dan aplikasi bisa memakai SMA, WMA, atau ARIMA sesuai pilihan sistem.


In [16]:
%%writefile prediksi_lead_time.py
"""Use an exported Wijaya Kusuma LightGBM model for an exact lead time."""

from datetime import date, timedelta
from pathlib import Path
import pickle

import numpy as np
import pandas as pd


def load_lead_time_model(model_dir, lead_time_days):
    """Load a model from the application's trusted model directory."""
    if type(lead_time_days) is not int or lead_time_days <= 0:
        raise ValueError("Lead time harus berupa bilangan bulat positif.")
    model_path = Path(model_dir) / f"lightgbm_{lead_time_days}d.pkl"
    if not model_path.is_file():
        raise ValueError(f"Model LightGBM untuk lead time {lead_time_days} hari belum tersedia.")
    with model_path.open("rb") as handle:
        bundle = pickle.load(handle)
    metadata = bundle["metadata"]
    if metadata["horizon_days"] != lead_time_days or "lead_time" not in metadata.get("forecast_uses", []):
        raise ValueError(f"Model {model_path.name} belum dikonfigurasi untuk lead time {lead_time_days} hari.")
    return bundle


def predict_lead_time(bundle, forecast_date, features):
    """Predict total units for one or more groups from features available at the forecast date."""
    metadata = bundle["metadata"]
    horizon = metadata["horizon_days"]
    origin = date.fromisoformat(forecast_date)
    matrix = pd.DataFrame([features] if isinstance(features, dict) else features)
    missing = [name for name in metadata["feature_names"] if name not in matrix.columns]
    if matrix.empty or missing:
        raise ValueError(f"Fitur prediksi kosong atau tidak lengkap: {missing}.")
    matrix = matrix[metadata["feature_names"]].apply(pd.to_numeric, errors="raise").astype(float)
    required = [name for name in matrix.columns if name not in metadata["nullable_features"]]
    if matrix[required].isna().any().any() or np.isinf(matrix.to_numpy()).any():
        raise ValueError("Fitur prediksi berisi nilai kosong atau tak terhingga yang tidak didukung.")
    for column, levels in metadata["category_levels"].items():
        if not matrix[column].isin(levels.values()).all():
            raise ValueError(f"Kode fitur {column} tidak ditemukan di metadata model.")
    if not matrix["karat"].isin(metadata["known_karat"]).all():
        raise ValueError("Kadar tidak ditemukan di metadata model.")
    predictions = np.maximum(0.0, bundle["model"].predict(matrix, num_threads=1))
    return [
        {
            "forecast_date": origin.isoformat(),
            "forecast_start": (origin + timedelta(days=1)).isoformat(),
            "forecast_end": (origin + timedelta(days=horizon)).isoformat(),
            "lead_time_days": horizon,
            "predicted_units": float(units),
        }
        for units in predictions
    ]


Writing prediksi_lead_time.py


In [17]:
from math import ceil
import prediksi_lead_time

# Gunakan tanggal dan fitur dari contoh data uji agar bisa langsung dijalankan.
examples_lead_time = []
for lead_time_days in sorted(set(LEAD_TIMES)):
    bundle = prediksi_lead_time.load_lead_time_model(output_dir, lead_time_days)
    metadata = bundle["metadata"]
    first_origin = split_origins[lead_time_days]["uji"][0]
    forecast_date = dates[first_origin].date().isoformat()
    row = parts_by_horizon[lead_time_days]["uji"][0].iloc[0]
    features = {
        name: None if pd.isna(row[name]) else float(row[name])
        for name in metadata["feature_names"]
    }
    result = prediksi_lead_time.predict_lead_time(bundle, forecast_date, features)[0]
    safety_stock = 2  # Contoh. Gunakan nilai konfigurasi dari sistem saat integrasi.
    example = {
        "group": {**identities.iloc[0].to_dict(), "kadar": int(identities.iloc[0]["kadar"])},
        "input": {"lead_time_days": lead_time_days, "forecast_date": forecast_date, "features": features},
        "output": result,
        "example_rop": {"safety_stock": safety_stock, "rop": ceil(result["predicted_units"] + safety_stock)},
    }
    examples_lead_time.append(example)
    print(json.dumps(result, ensure_ascii=False, indent=2))

(output_dir / "contoh_prediksi_lead_time.json").write_text(
    json.dumps(examples_lead_time, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
(output_dir / "prediksi_lead_time.py").write_bytes(Path(prediksi_lead_time.__file__).read_bytes())


{
  "forecast_date": "2025-10-01",
  "forecast_start": "2025-10-02",
  "forecast_end": "2025-10-15",
  "lead_time_days": 14,
  "predicted_units": 12.32748525673123
}


2722

## 13. Unduh semua hasil

Sel ini mengemas model, metadata, dan hasil evaluasi ke satu file ZIP untuk diunduh.


In [18]:
from shutil import make_archive

archive = make_archive("/content/Hasil_LightGBM_ARIMA_Wijaya_Kusuma_Lead_Time_dan_Proyeksi", "zip", output_dir)
files.download(archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Referensi metodologi

- [Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*](https://otexts.com/fpp3/tscv.html) menjelaskan evaluasi deret waktu dengan tanggal ramalan bergulir.
- [Dokumentasi ARIMA](https://stat.ethz.ch/R-manual/R-devel/library/stats/html/arima.html) menjelaskan metode estimasi *conditional sum of squares*.
